# Orion-XAI QLoRA v17
## Fix: patch accelerate/optimizer.py train()/eval() to be conditional (AdamW has no .train() method)

In [ ]:
import subprocess, sys, os

def run(cmd, label=''):
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print('STDOUT:', r.stdout[-800:])
        print('STDERR:', r.stderr[-800:])
        raise RuntimeError('Failed: ' + label)
    print(label, 'OK')

# Step 1: HF stack
run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers==4.45.0', 'peft==0.11.1', 'trl==0.9.6',
    'accelerate==0.30.1', 'datasets==2.19.1',
    '--upgrade'
], 'HF stack')

# Step 2: force-reinstall torch 2.3.1+cu121 (sm_60 support dropped in 2.4+)
run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torch==2.3.1+cu121', 'torchvision==0.18.1+cu121',
    '--index-url', 'https://download.pytorch.org/whl/cu121',
    '--force-reinstall'
], 'torch 2.3.1+cu121')

# Step 3: force-reinstall bitsandbytes 0.44.1 with --no-deps
run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'bitsandbytes==0.44.1', '--force-reinstall', '--no-deps'
], 'bitsandbytes 0.44.1 --no-deps')

# Verify versions
for pkg in ('torch', 'bitsandbytes'):
    r = subprocess.run([sys.executable, '-m', 'pip', 'show', pkg],
                       capture_output=True, text=True)
    for ln in r.stdout.splitlines():
        if ln.startswith('Version') or ln.startswith('Location'):
            print(pkg + ':', ln)

# Step 4: remove packages that open import paths into torch._dynamo
subprocess.run([sys.executable, '-m', 'pip', 'uninstall',
               'torchao', 'flax', 'jax', 'jaxlib', '-y'], capture_output=True)
print('torchao + jax/flax removed')

# Step 5: patch torch/_dynamo/utils.py
# Match the dict-KEY line "    np.random: tnp.random," by checking stripped.startswith
import torch as _t
_dynamo_utils = os.path.join(os.path.dirname(_t.__file__), '_dynamo', 'utils.py')
if os.path.exists(_dynamo_utils):
    with open(_dynamo_utils, 'r') as f:
        _lines = f.readlines()
    _new = []
    _patched = False
    for _ln in _lines:
        _stripped = _ln.lstrip()
        if (_stripped.startswith('np.random') and ':' in _stripped
                and not _stripped.startswith('#')):
            _indent = _ln[:len(_ln) - len(_stripped)]
            _new.append(_indent + '# ' + _stripped)
            _patched = True
        else:
            _new.append(_ln)
    if _patched:
        with open(_dynamo_utils, 'w') as f:
            f.writelines(_new)
        print('torch/_dynamo/utils.py patched OK')
del _t

# Step 6: patch fsdp_utils.py
r = subprocess.run([sys.executable, '-m', 'pip', 'show', 'accelerate'],
                   capture_output=True, text=True)
accel_loc = None
for line in r.stdout.splitlines():
    if line.startswith('Location:'):
        accel_loc = line.split(':', 1)[1].strip(); break
if accel_loc:
    fsdp_path = os.path.join(accel_loc, 'accelerate', 'utils', 'fsdp_utils.py')
    if os.path.exists(fsdp_path):
        with open(fsdp_path, 'r') as f:
            fc = f.read()
        for old in [
            'if is_torch_version(">", FSDP_PYTORCH_VERSION) and is_torch_distributed_available():',
            'if is_torch_version(">=", FSDP_PYTORCH_VERSION) and is_torch_distributed_available():'
        ]:
            if old in fc:
                fc = fc.replace(old, 'if False:  # patched')
                with open(fsdp_path, 'w') as f:
                    f.write(fc)
                print('fsdp_utils.py patched OK'); break

    # Step 7: patch accelerate/optimizer.py
    # accelerate 0.30.1 calls self.optimizer.train() / self.optimizer.eval()
    # unconditionally, but torch.optim.AdamW has no train()/eval() methods.
    # Fix: make the calls conditional (supports schedule_free but safe for AdamW)
    opt_path = os.path.join(accel_loc, 'accelerate', 'optimizer.py')
    if os.path.exists(opt_path):
        with open(opt_path, 'r') as f:
            oc = f.read()
        opt_patched = False
        for meth, check in [
            ('return self.optimizer.train()',
             'return self.optimizer.train() if hasattr(self.optimizer, "train") and callable(self.optimizer.train) else None'),
            ('return self.optimizer.eval()',
             'return self.optimizer.eval() if hasattr(self.optimizer, "eval") and callable(self.optimizer.eval) else None'),
        ]:
            if meth in oc:
                oc = oc.replace(meth, check)
                opt_patched = True
        if opt_patched:
            with open(opt_path, 'w') as f:
                f.write(oc)
            print('accelerate/optimizer.py patched OK')
        else:
            print('accelerate/optimizer.py: no patch needed')

# Step 8: patch bitsandbytes/nn/__init__.py
r = subprocess.run([sys.executable, '-m', 'pip', 'show', 'bitsandbytes'],
                   capture_output=True, text=True)
bnb_loc = None
bnb_ver = '?'
for ln in r.stdout.splitlines():
    if ln.startswith('Location:'): bnb_loc = ln.split(':', 1)[1].strip()
    if ln.startswith('Version:'): bnb_ver = ln.split(':', 1)[1].strip()
print('bitsandbytes', bnb_ver, '@', bnb_loc)

if bnb_loc:
    nn_init = os.path.join(bnb_loc, 'bitsandbytes', 'nn', '__init__.py')
    if os.path.exists(nn_init):
        with open(nn_init, 'r') as f:
            lines = f.readlines()
        NL = chr(10)
        new_lines = []
        i = 0
        patched = False
        while i < len(lines):
            ln = lines[i]
            if ('from .triton_based_modules import' in ln
                    and not ln.lstrip().startswith('#')):
                block = [ln]
                open_p = ln.count('(') - ln.count(')')
                i += 1
                while open_p > 0 and i < len(lines):
                    block.append(lines[i])
                    open_p += lines[i].count('(') - lines[i].count(')')
                    i += 1
                new_lines.append('try:' + NL)
                for bl in block:
                    new_lines.append('    ' + bl)
                new_lines.append('except (ImportError, ModuleNotFoundError):' + NL)
                new_lines.append('    pass  # triton.ops not available' + NL)
                patched = True
            else:
                new_lines.append(ln)
                i += 1
        if patched:
            with open(nn_init, 'w') as f:
                f.writelines(new_lines)
            print('bitsandbytes/nn/__init__.py patched OK')
        else:
            print('bitsandbytes/nn/__init__.py: OK (no triton import)')

import numpy, torch
print('numpy:', numpy.__version__, '| torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())


In [ ]:
import os, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

os.environ['HF_TOKEN'] = 'hf_...YOUR_TOKEN_HERE...'
os.environ['HUGGING_FACE_HUB_TOKEN'] = 'hf_...YOUR_TOKEN_HERE...'

if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('GPU:', props.name, '| sm_%d%d | %.1f GB' % (
        props.major, props.minor, props.total_memory/1e9))

MODEL_NAME = 'meta-llama/Llama-3.2-3B-Instruct'
MAX_SEQ_LENGTH = 2048
COMPUTE_DTYPE = torch.float16

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=COMPUTE_DTYPE,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME, trust_remote_code=True, token=os.environ['HF_TOKEN'])
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

print('Loading model...')
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto',
    torch_dtype=COMPUTE_DTYPE,
    trust_remote_code=True,
    token=os.environ['HF_TOKEN'],
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_config = LoraConfig(
    r=16, lora_alpha=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
print('Model + LoRA ready!')


In [ ]:
import json, os, glob
from datasets import Dataset

DATASET_PATH = '/kaggle/input/orion-xai-dataset/orion_xai_final_dataset.jsonl'
if not os.path.exists(DATASET_PATH):
    files = glob.glob('/kaggle/input/**/*.jsonl', recursive=True)
    DATASET_PATH = files[0] if files else DATASET_PATH
    print('Using:', DATASET_PATH)

EOS_TOKEN = tokenizer.eos_token
TEMPLATE = ('Below is an instruction that describes a task. '
            'Write a response that appropriately completes the request.\n\n'
            '### Instruction:\n{instruction}\n\n### Response:\n{output}')

records = []
with open(DATASET_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            ex = json.loads(line)
            records.append({'text': TEMPLATE.format(
                instruction=ex.get('instruction', ''),
                output=ex.get('output', '')) + EOS_TOKEN})

dataset = Dataset.from_list(records)
print('Dataset: %d examples' % len(dataset))


In [ ]:
from transformers import TrainingArguments
from trl import SFTTrainer

training_args = TrainingArguments(
    output_dir='/kaggle/working/orion-xai-checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_ratio=0.05,
    optim='adamw_torch',
    fp16=True, bf16=False,
    logging_steps=10,
    save_strategy='epoch',
    save_total_limit=1,
    report_to='none',
    max_grad_norm=0.3,
    weight_decay=0.01,
    seed=42,
    dataloader_num_workers=0,
)

trainer = SFTTrainer(
    model=model, tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field='text',
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False, args=training_args,
)

print('Starting training...')
stats = trainer.train()
print('Done! Runtime: %.0fs | Loss: %.4f' % (
    stats.metrics['train_runtime'], stats.metrics['train_loss']))


In [ ]:
import os
ADAPTER_PATH = '/kaggle/working/orion-xai-lora'
os.makedirs(ADAPTER_PATH, exist_ok=True)
model.save_pretrained(ADAPTER_PATH)
tokenizer.save_pretrained(ADAPTER_PATH)
print('Saved to:', ADAPTER_PATH)
for fn in sorted(os.listdir(ADAPTER_PATH)):
    print('  %s: %.2f MB' % (fn, os.path.getsize(os.path.join(ADAPTER_PATH, fn))/1e6))


In [ ]:
import torch
TEST = 'Explain how SHAP values work for explaining ML model predictions in a cybersecurity SOC context.'
prompt = ('Below is an instruction that describes a task. Write a response that '
          'appropriately completes the request.\n\n'
          '### Instruction:\n' + TEST + '\n\n### Response:\n')
model.eval()
inputs = tokenizer(prompt, return_tensors='pt').to('cuda')
with torch.no_grad():
    out = model.generate(**inputs, max_new_tokens=300, temperature=0.7,
                         top_p=0.9, do_sample=True,
                         pad_token_id=tokenizer.eos_token_id)
print(tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True))
print('Orion-XAI training complete!')
